# Q-ErrorID: noise models and diagnostics

This notebook demonstrates the physical core without QNN, Haiqu, or UI dependencies. The learned quantities are local generator parameters rather than arbitrary Kraus matrices.

In [ ]:
import sys
from pathlib import Path

import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from q_error_id.core import (
    ReadoutConfusion,
    analyze_identifiability,
    build_channel,
    channel_to_choi,
    channel_to_ptm,
    extract_features,
    one_qubit_parameters,
    one_qubit_protocol,
    representative_parameters,
    two_qubit_protocol,
    validate_channel,
)

## One-qubit mixed channel

The finite-time channel is obtained as $\mathcal E=\exp(\mathcal L)$, so nonnegative Lindblad rates produce a CPTP map up to numerical roundoff.

In [ ]:
parameters_1q = one_qubit_parameters(
    alpha=np.array([0.05, -0.03, 0.02]),
    gamma=np.array([0.010, 0.014, 0.007]),
    kappa_down=0.022,
)
channel_1q = build_channel(parameters_1q)
validation_1q = validate_channel(channel_1q)
ptm_1q = channel_to_ptm(channel_1q)
choi_state_1q = channel_to_choi(channel_1q)
validation_1q, ptm_1q, np.trace(choi_state_1q)

## Protocol B and finite shots

The one-qubit bank contains six Pauli eigenstates and three observables, giving 18 scalar features. Readout confusion changes the measured features but never the physical target labels.

In [ ]:
protocol_1q = one_qubit_protocol()
exact = extract_features(channel_1q, protocol_1q)
confusion = ReadoutConfusion(0.018, 0.024)
shot_1024 = extract_features(
    channel_1q, protocol_1q, shots=1024,
    rng=np.random.default_rng(7), readout_confusion=confusion,
)
list(zip(protocol_1q.feature_labels[:6], exact[:6], shot_1024[:6]))

## Local identifiability

A full-rank numerical Jacobian means that all generator parameters are locally distinguishable at the reference point. The condition number quantifies sensitivity to shot noise and model mismatch.

In [ ]:
report_1q = analyze_identifiability(
    representative_parameters('1Q'), protocol_1q
)
protocol_2q = two_qubit_protocol(target_features=80)
report_2q = analyze_identifiability(
    representative_parameters('CX', basis=('ZI', 'IZ', 'ZX', 'ZZ')),
    protocol_2q,
)
{
    '1q': (report_1q.feature_count, report_1q.rank, report_1q.condition_number),
    '2q': (report_2q.feature_count, report_2q.rank, report_2q.condition_number),
}

The two-qubit minimal reliable subset contains ten settings because this validation includes four coherent rates, four stochastic rates, and two known-local-channel damping values. If the local damping values are fixed inputs from the one-qubit calibration stage, call the Jacobian tools with `include_kappa=False`; the gate-specific target then has eight parameters.